In [15]:
# ── Cell 1: Setup ─────────────────────────────────────────────
import os, sys, time, json
import torch
import warnings
warnings.filterwarnings('ignore')

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools.retriever import create_retriever_tool
from langchain_chroma import Chroma
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from sentence_transformers import CrossEncoder
from typing import TypedDict, Annotated, List
import operator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHROMA_DB_PATH = '../data/chroma_db'
os.environ['TAVILY_API_KEY'] = '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'

print(f'Device        : {DEVICE}')
print(f'Torch version : {torch.__version__}')
print('Setup complete')

Device        : cpu
Torch version : 2.6.0+cu124
Setup complete


In [16]:
# ── Cell 2: Load models ────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = Chroma(persist_directory=CHROMA_DB_PATH, embedding_function=embeddings)
llm = ChatOllama(model='llama3.1:8b', temperature=0, num_ctx=4096)
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')
web_tool = TavilySearch(max_results=3, topic='general')

print(f'Chunks in DB  : {vectorstore._collection.count()}')
print('All models loaded')

Chunks in DB  : 211190
All models loaded


In [17]:
# ── Cell 3: Agent definition ──
def rerank_docs(query, docs, k=3):
    if not docs: return docs
    pairs = [[query, d.page_content] for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:k]]

class AgentState(TypedDict):
    messages    : Annotated[List, operator.add]
    question    : str
    context     : str
    answer      : str
    search_type : str
    iterations  : int

META_KEYWORDS = ['previous','last','before','earlier','first','asked','said','what did i','what was my','our conversation','first prompt']

def retrieve_node(state):
    q = state['question']
    if any(w in q.lower() for w in META_KEYWORDS):
        return {**state, 'context': '', 'search_type': 'memory'}
    docs = vectorstore.similarity_search(q, k=10)
    docs = rerank_docs(q, docs, k=3)
    context = '\n\n'.join([f"[Source: {d.metadata['title']}]\n{d.page_content}" for d in docs])
    return {**state, 'context': context, 'search_type': 'local'}

def grade_node(state):
    if state['search_type'] == 'memory':
        return {**state, 'search_type': 'memory'}
    prompt = ChatPromptTemplate.from_template(
        'Does this context contain information to answer the question?\nQuestion: {question}\nContext: {context}\nReply with ONLY one word: yes or no')
    grade = (prompt | llm | StrOutputParser()).invoke({'question': state['question'], 'context': state['context'][:800]}).strip().lower()
    return {**state, 'search_type': 'local' if 'yes' in grade else 'web_needed'}

def web_search_node(state):
    results = web_tool.invoke(state['question'])
    if isinstance(results, list):
        context = '\n\n'.join([f"[Web: {r.get('url','')}]\n{r.get('content','')}" for r in results])
    else:
        context = str(results)
    return {**state, 'context': context, 'search_type': 'web'}

def generate_node(state):
    system_msg = SystemMessage(content='You are a helpful conversational assistant. Answer using the provided context. Use conversation history for meta-questions. Stay focused on the current question only.')
    q = state['question']
    history = state.get('messages', [])
    is_meta = any(w in q.lower() for w in META_KEYWORDS)
    if is_meta:
        hist_text = '\n'.join([f"{'User' if isinstance(m,HumanMessage) else 'Bot'}: {m.content}" for m in history])
        ctx_msg = HumanMessage(content=f'Conversation history:\n{hist_text}\n\nQuestion: {q}\nAnswer from history only:')
        all_msgs = [system_msg, ctx_msg]
    else:
        ctx_msg = HumanMessage(content=f'Context:\n{state["context"]}\n\nQuestion: {q}\nAnswer from context:')
        all_msgs = [system_msg] + history[-4:] + [ctx_msg]
    response = llm.invoke(all_msgs)
    return {**state, 'answer': response.content}

def critique_node(state):
    prompt = ChatPromptTemplate.from_template(
        'Is this answer grounded in the context?\nQuestion: {question}\nContext: {context}\nAnswer: {answer}\nReply ONLY: good or retry')
    result = (prompt | llm | StrOutputParser()).invoke({'question': state['question'], 'context': state['context'][:500], 'answer': state['answer'][:500]}).strip().lower()
    return {**state, 'iterations': state.get('iterations', 0) + 1}

def route_after_grade(state):
    if state['search_type'] in ('memory',): return 'generate'
    if state['search_type'] == 'web_needed': return 'web_search'
    return 'generate'

def route_after_critique(state):
    if state['iterations'] >= 1: return END
    if not state.get('answer','').strip(): return 'web_search'
    return END

graph = StateGraph(AgentState)
graph.add_node('retrieve',   retrieve_node)
graph.add_node('grade',      grade_node)
graph.add_node('web_search', web_search_node)
graph.add_node('generate',   generate_node)
graph.add_node('critique',   critique_node)
graph.set_entry_point('retrieve')
graph.add_edge('retrieve', 'grade')
graph.add_conditional_edges('grade', route_after_grade, {'web_search':'web_search','generate':'generate'})
graph.add_edge('web_search', 'generate')
graph.add_edge('generate', 'critique')
graph.add_conditional_edges('critique', route_after_critique, {'web_search':'web_search', END:END})
memory = MemorySaver()
agent  = graph.compile(checkpointer=memory)
print('Agent compiled successfully')

Agent compiled successfully


In [18]:
# ── Cell 4: Chat helper ────────────────────────────────────────
results_log = []

def chat(question, thread_id='default', verbose=True):
    config = {'configurable': {'thread_id': thread_id}}
    state = {'messages': [HumanMessage(content=question)], 'question': question,
             'context': '', 'answer': '', 'search_type': 'local', 'iterations': 0}
    t0 = time.time()
    result = agent.invoke(state, config=config)
    latency = time.time() - t0
    entry = {'question': question, 'answer': result['answer'],
             'search_type': result['search_type'], 'latency_s': round(latency, 2)}
    results_log.append(entry)
    if verbose:
        print(f'\n{"="*65}')
        print(f'Q: {question}')
        print(f'Route: {result["search_type"]}  |  Latency: {latency:.1f}s')
        print(f'{"─"*65}')
        print(f'A: {result["answer"]}')
    return result['answer']

In [19]:
# ── Cell 5: EVAL 1 — Single-hop factual queries (local KB) ────
print('='*65)
print('EVALUATION 1: Single-hop Factual Queries (Local Knowledge Base)')
print('Expected route: local | Source: Wikipedia chunks')
print('='*65)

factual_queries = [
    'What is the Hall effect?',
    'What is quantum tunneling?',
    'Explain the theory of relativity.',
    'What causes earthquakes?',
    'What is photosynthesis?',
]

for q in factual_queries:
    chat(q, thread_id='eval1')

EVALUATION 1: Single-hop Factual Queries (Local Knowledge Base)
Expected route: local | Source: Wikipedia chunks

Q: What is the Hall effect?
Route: local  |  Latency: 26.5s
─────────────────────────────────────────────────────────────────
A: The Hall effect is a phenomenon where a voltage develops across conductors transverse to an electric current in the conductor and magnetic field perpendicular to the current.

Q: What is quantum tunneling?
Route: web  |  Latency: 87.7s
─────────────────────────────────────────────────────────────────
A: Quantum tunneling refers to a phenomenon in which particles appear on the other side of an energy barrier without having enough classical energy to cross it. It happens when a particle's wave function, which represents the probability of its location, extends into and beyond a potential energy barrier. The process doesn't involve the particle physically breaking the barrier, but rather bypassing it through quantum tunneling probability.

Q: Explain

In [20]:
# ── Cell 6: EVAL 2 — Web fallback (current events) ────────────
print('='*65)
print('EVALUATION 2: Web Fallback — Current Events')
print('Expected route: web | Trigger: grade=no on local KB')
print('='*65)

web_queries = [
    'What are the latest AI models released in 2025?',
    'What is the current condition of the Strait of Hormuz?',
    'Who won the most recent FIFA World Cup?',
]

for q in web_queries:
    chat(q, thread_id='eval2')

EVALUATION 2: Web Fallback — Current Events
Expected route: web | Trigger: grade=no on local KB

Q: What are the latest AI models released in 2025?
Route: web  |  Latency: 65.1s
─────────────────────────────────────────────────────────────────
A: The latest AI models released in 2025 include:

* GPT-5 and its variant GPT-5.1
* Google Gemini 3
* Anthropic Claude 4 (Opus & Sonnet)
* Meta Llama 4 (Scout)

Additionally, other notable models mentioned in the context are:

* Optibot
* Claude 3 Opus
* Gemini
* Grok
* Perplexity

These models have been reported to outperform previous versions and offer improved capabilities.

Q: What is the current condition of the Strait of Hormuz?
Route: web  |  Latency: 52.5s
─────────────────────────────────────────────────────────────────
A: I don't have any information about AI models released in 2025. However, I can answer your original question.

According to the provided context, the current condition of the Strait of Hormuz is that it is now fully op

In [21]:
# ── Cell 7: EVAL 3 — Multi-turn conversational memory ─────────
print('='*65)
print('EVALUATION 3: Multi-turn Conversational Memory')
print('Tests: follow-up questions, pronoun resolution, history recall')
print('='*65)

# Thread keeps conversation history
chat('Tell me about Albert Einstein.', thread_id='memory_test')
chat('What was his most famous equation?', thread_id='memory_test')
chat('When did he win the Nobel Prize?', thread_id='memory_test')
chat('What was the first question I asked you?', thread_id='memory_test')

EVALUATION 3: Multi-turn Conversational Memory
Tests: follow-up questions, pronoun resolution, history recall

Q: Tell me about Albert Einstein.
Route: local  |  Latency: 46.5s
─────────────────────────────────────────────────────────────────
A: Albert Einstein was a German-born theoretical physicist who is widely held to be one of the greatest and most influential scientists of all time. He is best known for developing the theory of relativity, which revolutionized our understanding of space and time. In addition to his work on relativity, Einstein also made important contributions to quantum mechanics.

Q: What was his most famous equation?
Route: web  |  Latency: 67.0s
─────────────────────────────────────────────────────────────────
A: E=MC^2 is Albert Einstein's most famous equation. It relates the concept of mass and energy, showing that a small amount of mass can be converted into a large amount of energy, and vice versa. This equation has had a significant impact on our underst

'The first question you asked me was "What was his most famous equation?"'

In [22]:
# ── Cell 8: EVAL 4 — Multi-hop / multi-step query ─────────────
print('='*65)
print('EVALUATION 4: Multi-hop / Multi-step Query')
print('Tests: combining local retrieval + web + conversation history')
print('='*65)

# Step 1: local knowledge
chat('Explain how neural networks learn using backpropagation.', thread_id='multihop')

# Step 2: web knowledge
chat('What are the most powerful neural network architectures available in 2025?', thread_id='multihop')

# Step 3: multi-hop — combines both previous answers
chat('Based on what you told me about backpropagation and the 2025 architectures, \
which modern architecture would benefit most from improved training algorithms?', thread_id='multihop')

EVALUATION 4: Multi-hop / Multi-step Query
Tests: combining local retrieval + web + conversation history

Q: Explain how neural networks learn using backpropagation.
Route: local  |  Latency: 41.9s
─────────────────────────────────────────────────────────────────
A: According to the provided context, neural networks learn to model complex relationships between inputs and outputs and find patterns in data. The most common training technique is the backpropagation algorithm, which uses local search to choose the weights that will get the right output for each input during training. In theory, a neural network can learn any function using this method.

Q: What are the most powerful neural network architectures available in 2025?
Route: web  |  Latency: 98.5s
─────────────────────────────────────────────────────────────────
A: Based on the provided context, the most powerful neural network architectures available in 2025 are:

1. Transformers: According to the article "Transformers: The Po

"To answer your question, I'll need to recall my previous explanations.\n\nFirstly, let's review how neural networks learn using backpropagation. Backpropagation is a key component of the supervised learning process in neural networks. It's an algorithm used to train artificial neural networks by minimizing the error between the network's predictions and the actual outputs. Here's a simplified overview:\n\n1.  **Forward Pass**: The input data flows through the network, and each layer processes it until the output is generated.\n2.  **Error Calculation**: The difference between the predicted output and the actual output (error) is calculated for each example in the training dataset.\n3.  **Backward Pass**: The error is propagated backwards through the network, adjusting the weights of each connection based on how much they contributed to the overall error.\n\nNow, let's discuss the most powerful neural network architectures available in 2025. Unfortunately, I don't have specific informa

In [23]:
# ── Cell 9: EVAL 5 — Hallucination analysis ───────────────────
print('='*65)
print('EVALUATION 5: Hallucination Analysis')
print('Tests grounding, source-awareness, uncertainty acknowledgement')
print('='*65)

# Test 1: Should answer from context, not hallucinate
print('\n--- Test 1: Grounding check (should cite Wikipedia source) ---')
chat('What is the Bernoulli principle and where is it used?', thread_id='halluc_test')

# Test 2: Should acknowledge lack of info, not fabricate
print('\n--- Test 2: Unknown entity (should say insufficient info) ---')
chat('Who is Zara Qasim and what did she invent in 2024?', thread_id='halluc_test')

# Test 3: Ambiguous — should use web, not hallucinate recent stats
print('\n--- Test 3: Recent statistics (should use web) ---')
chat('What is the current global adoption rate of electric vehicles in 2025?', thread_id='halluc_test')

EVALUATION 5: Hallucination Analysis
Tests grounding, source-awareness, uncertainty acknowledgement

--- Test 1: Grounding check (should cite Wikipedia source) ---

Q: What is the Bernoulli principle and where is it used?
Route: local  |  Latency: 44.4s
─────────────────────────────────────────────────────────────────
A: The Bernoulli principle is a fundamental relationship between pressure, density, and flow velocity for incompressible flow. It provides one method for calculating aerodynamic lift. This principle is used in aerodynamics to understand how air resistance affects objects in motion.

--- Test 2: Unknown entity (should say insufficient info) ---

Q: Who is Zara Qasim and what did she invent in 2024?
Route: web  |  Latency: 98.5s
─────────────────────────────────────────────────────────────────
A: There is no information about Zara Qasim inventing something in 2024. The search results mention her education, work experience, and skills as a dental hygienist/therapist or psych

'The current global adoption rate of electric vehicles in 2025 is expected to increase by 25% globally, with over 4 million EVs sold worldwide in the first quarter. This growth rate is similar to that of 2024.'

In [24]:
# ── Cell 10: Evaluation summary table ─────────────────────────
import pandas as pd

df = pd.DataFrame(results_log)
df['answer_len'] = df['answer'].str.len()

print('='*65)
print('EVALUATION SUMMARY')
print('='*65)
print(f'Total queries evaluated : {len(df)}')
print(f'Local KB route          : {(df.search_type=="local").sum()}')
print(f'Web fallback route      : {(df.search_type=="web").sum()}')
print(f'Memory route            : {(df.search_type=="memory").sum()}')
print(f'Avg latency (s)         : {df.latency_s.mean():.1f}')
print(f'Avg answer length       : {df.answer_len.mean():.0f} chars')
print()
print('Route distribution:')
print(df['search_type'].value_counts())

# Save results
df[['question','search_type','latency_s','answer_len']].to_csv('../data/processed/eval_results.csv', index=False)
print('\nResults saved to ../data/processed/eval_results.csv')

print('\n--- Per-query results ---')
for _, row in df.iterrows():
    print(f"[{row.search_type:6s}] [{row.latency_s:4.1f}s] {row.question[:60]}")

EVALUATION SUMMARY
Total queries evaluated : 18
Local KB route          : 8
Web fallback route      : 9
Memory route            : 1
Avg latency (s)         : 74.3
Avg answer length       : 554 chars

Route distribution:
search_type
web       9
local     8
memory    1
Name: count, dtype: int64

Results saved to ../data/processed/eval_results.csv

--- Per-query results ---
[local ] [26.5s] What is the Hall effect?
[web   ] [87.7s] What is quantum tunneling?
[local ] [77.8s] Explain the theory of relativity.
[web   ] [52.6s] What causes earthquakes?
[local ] [66.0s] What is photosynthesis?
[web   ] [65.1s] What are the latest AI models released in 2025?
[web   ] [52.5s] What is the current condition of the Strait of Hormuz?
[local ] [45.7s] Who won the most recent FIFA World Cup?
[local ] [46.5s] Tell me about Albert Einstein.
[web   ] [67.0s] What was his most famous equation?
[web   ] [74.2s] When did he win the Nobel Prize?
[memory] [261.0s] What was the first question I asked you?
[lo

In [25]:
# ── Cell 11: Retrieval quality analysis ───────────────────────
print('='*65)
print('RETRIEVAL QUALITY ANALYSIS')
print('Reranker scores for representative queries')
print('='*65)

test_queries = [
    'What is machine learning?',
    'How does quantum tunneling work?',
    'What is the Hall effect?',
    'Explain photosynthesis',
    'What causes earthquakes?',
]

from sentence_transformers import CrossEncoder
reranker_eval = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

print(f'{"Query":<40} {"Top-1":>8} {"Top-2":>8} {"Top-3":>8} {"Source"}')
print('-'*80)

for q in test_queries:
    docs  = vectorstore.similarity_search(q, k=10)
    pairs = [[q, d.page_content] for d in docs]
    scores = reranker_eval.predict(pairs)
    top3_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
    top3_scores = [scores[i] for i in top3_idx]
    top_source = docs[top3_idx[0]].metadata.get('title', 'N/A')
    print(f"{q[:38]:<40} {top3_scores[0]:>8.3f} {top3_scores[1]:>8.3f} {top3_scores[2]:>8.3f}  {top_source[:25]}")

RETRIEVAL QUALITY ANALYSIS
Reranker scores for representative queries
Query                                       Top-1    Top-2    Top-3 Source
--------------------------------------------------------------------------------
What is machine learning?                   0.999    0.995    0.693  Brain
How does quantum tunneling work?            0.996    0.743    0.316  Atom
What is the Hall effect?                    0.926    0.658    0.658  Condensed matter physics
Explain photosynthesis                      0.819    0.573    0.566  Additive synthesis
What causes earthquakes?                    0.991    0.466    0.322  Attribution of recent cli
